# Group 8 — ISIC skin-lesion classification

Coding component skeleton. Every requirement from the client document maps to one section below.
Target runtime: Google Colab, single TensorFlow / Keras stack.
Fill in the TODO blocks in order; each section must stay runnable from top to bottom.

**Datasets (both are used)**
- ISIC 2018 Task 3 (HAM10000, 7 classes) — train / validation / internal test, lesion-disjoint split
- ISIC 2019 (8 classes) — external validation, SCC excluded, hash-deduped against 2018

The third dataset listed in the client document (SIIM-ISIC 2020) is not used: its archive is over 100 GB.
This deviation is recorded in `AGENTS.md` and must be stated in the report.

**Fixed label space (7 classes)**: MEL, NV, BCC, AKIEC, BKL, DF, VASC


## Requirement map

| Section | Requirement | Main artifact |
|---|---|---|
| 0 | setup | paths, output folders, dataset file counts |
| 1 | 2.1 | library version table |
| 2 | 2.2 | gradient tape demonstration |
| 3 | 2.3 | training vs evaluation memory |
| data | foundation | `isic2018` / `isic2019` tables |
| 5 | 2.4 | preprocessing and label encoding |
| 6 | 2.5–2.6 | augmentation figure |
| 7 | 2.7–2.9 | `model.summary()`, feature maps |
| 8 | 1.1 | gradient-magnitude figure, VGG vs ResNet |
| 9 | 2.10–2.13 | training history and learning curves |
| 10 | 2.14–2.17 | confusion matrix, top-3 confident errors |
| 11 | 2.18–2.20 | external results, unseen image, calibration |
| 12 | submission | saved artifacts and results archive |

Coverage checklist: 1.1 · 2.1 · 2.2 · 2.3 · 2.4 · 2.5 · 2.6 · 2.7 · 2.8 · 2.9 · 2.10 · 2.11 ·
2.12 · 2.13 · 2.14 · 2.15 · 2.16 · 2.17 · 2.18 · 2.19 · 2.20


## Run plan

- **Once, before running anything**: download the two Kaggle datasets and unpack them into `data/`
  (the exact folder names are listed in section 0).
- **Session 1**: sections 0 → data layer; the printed counts must be larger than zero.
- **Session 2**: sections 2.1 → 2.9.
- **Session 3**: sections 2.10 → 2.13 (training, checkpoint each epoch).
- **Session 4**: sections 2.14 → 2.20 (evaluation and figures).


## 0. Setup and runtime

Download the two datasets from Kaggle **once** and unpack them into this project like this:

```
data/
  ISIC 2018/   HAM10000_metadata.csv + *.jpg   (HAM10000, 7 classes, 10,015 images)
  ISIC 2019/   ISIC_2019_Training_GroundTruth.csv + *.jpg
```

`data/` and `outputs/` are git-ignored, so the images never enter the repository.
The last line of the cell prints how many JPEGs are visible per dataset; a wrong folder name shows up
immediately as `0 jpg`.

Produces: fixed paths, output folders, seeds, dataset file counts.


In [ ]:
# Section 0 - paths and seeds. The two datasets are downloaded once from Kaggle and unpacked here:
#   data/ISIC 2018/  data/ISIC 2019/
import os, random, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

DATA_ROOT = Path("data")      # project folder; keep the same two subfolders on Colab
OUT_DIR = Path("outputs")     # figures and tables produced by later sections
(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

SEED = 8
random.seed(SEED)
np.random.seed(SEED)

assert DATA_ROOT.is_dir(), "create data/ with the two dataset folders first"
# First sanity check: every line must show a number larger than 0.
for name in ("ISIC 2018", "ISIC 2019"):
    print(f"{name:10s} {len(list((DATA_ROOT / name).rglob('*.jpg'))):6d} jpg")


## 2.1 Libraries

Required: list every library used and its version. Produces: printed version table saved to `env.json`.

In [ ]:
# TODO(2.1): import tensorflow, keras, numpy, pandas, scikit-learn, matplotlib, PIL, kaggle.
# TODO(2.1): record versions (python, tensorflow, keras, numpy, pandas, scikit-learn) to env.json.

## 2.2 Forward pass and automatic differentiation

Required: explain how the framework tracks the forward pass and recovers gradients. Produces: per-layer gradient norms for one real batch.

In [ ]:
# TODO(2.2): run one real training batch forward inside tf.GradientTape.
# TODO(2.2): call tape.gradient(loss, model.trainable_variables) and print per-layer gradient norms.
# TODO(2.2): write the markdown explanation of what the tape records and when gradients are released.

## 2.3 Training and evaluation memory

Required: explain the memory difference between training and evaluation mode. Produces: behaviour evidence and written comparison.

In [ ]:
# TODO(2.3): call the model with training=True and training=False on the same input and show the difference.
# TODO(2.3): show that evaluation needs no gradient tape (tf.stop_gradient / inference path).
# TODO(2.3): explain activation memory during training versus inference memory in the report.

## Data layer: load the two datasets from the project folder

No download code is needed: the folders above are read directly. Each dataset becomes one small table
with the columns the later sections need (`image`, `path`, `label` / `target`, `split`, `md5`).

Rules applied here:
- fixed 7-class label space; HAM10000 lowercase codes are mapped to class names;
- lesion-level split for ISIC 2018, so all images of one lesion stay in one split;
- SCC dropped from ISIC 2019 (the model has no SCC slot);
- duplicate images removed from ISIC 2019 by file md5 (it largely repeats ISIC 2018).

Produces: the `isic2018` and `isic2019` tables, plus `outputs/tables/isic2018_split.csv`.


In [ ]:
# 2.4 - ISIC 2018 (HAM10000): build image path + 7-class label + lesion id.
# HAM10000 stores lowercase codes such as "mel"; map them to the fixed class names.
CLASS_NAMES = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
DX_TO_CLASS = {"mel": "MEL", "nv": "NV", "bcc": "BCC", "akiec": "AKIEC",
               "bkl": "BKL", "df": "DF", "vasc": "VASC"}

base = DATA_ROOT / "ISIC 2018"
meta_path = next(base.rglob("HAM10000_metadata.csv"), None)
assert meta_path is not None, "data/ISIC 2018 must contain HAM10000_metadata.csv"
meta = pd.read_csv(meta_path)
jpgs = {p.stem: p for p in base.rglob("*.jpg")}     # image name -> full path

isic2018 = meta[["image_id", "lesion_id", "dx"]].rename(columns={"image_id": "image"})
isic2018["label"] = isic2018["dx"].map(DX_TO_CLASS)
isic2018["path"] = isic2018["image"].map(lambda name: str(jpgs[name]))
print(isic2018["label"].value_counts().to_string())


In [ ]:
# Lesion-level split: all images of one lesion land in the same split (no leakage between splits).
lesions = sorted(isic2018["lesion_id"].unique())
np.random.RandomState(SEED).shuffle(lesions)
val_lesions = set(lesions[int(len(lesions) * 0.70):int(len(lesions) * 0.85)])
test_lesions = set(lesions[int(len(lesions) * 0.85):])

def split_of(lesion_id):
    if lesion_id in val_lesions:
        return "validation"
    if lesion_id in test_lesions:
        return "test"
    return "train"

isic2018["split"] = isic2018["lesion_id"].map(split_of)
isic2018.to_csv(OUT_DIR / "tables" / "isic2018_split.csv", index=False)
print(isic2018["split"].value_counts().to_string())


In [ ]:
# 2.4 - ISIC 2019 is the external validation set. It has 8 classes; SCC has no slot in our 7 classes.
# The ground-truth CSV stores AKIEC as "AK", so the column names are mapped to the fixed class names.
ISIC2019_CLASSES = {"MEL": "MEL", "NV": "NV", "BCC": "BCC", "AK": "AKIEC",
                    "BKL": "BKL", "DF": "DF", "VASC": "VASC", "SCC": "SCC"}

base = DATA_ROOT / "ISIC 2019"
jpgs = {p.stem: p for p in base.rglob("*.jpg")}
truth_path = next(base.rglob("ISIC_2019_Training_GroundTruth.csv"), None)
assert truth_path is not None, "data/ISIC 2019 must contain ISIC_2019_Training_GroundTruth.csv"
truth = pd.read_csv(truth_path)
truth["label"] = truth[list(ISIC2019_CLASSES)].idxmax(axis=1).map(ISIC2019_CLASSES)
scc_count = int((truth["label"] == "SCC").sum())
isic2019 = truth[truth["label"] != "SCC"][["image", "label"]].copy()
isic2019["path"] = isic2019["image"].map(lambda name: str(jpgs[name]))
isic2019["split"] = "external_val"
print("isic2019:", len(isic2019), "| SCC dropped:", scc_count)


In [ ]:
# ISIC 2019 largely repeats ISIC 2018 images, so remove them: the external set must be unseen.
def md5(path):
    with open(path, "rb") as handle:
        return hashlib.md5(handle.read()).hexdigest()

isic2018["md5"] = [md5(p) for p in isic2018["path"]]
isic2019["md5"] = [md5(p) for p in isic2019["path"]]
duplicates = isic2019["md5"].isin(isic2018["md5"])
isic2019 = isic2019[~duplicates].reset_index(drop=True)
print("duplicate images removed from isic2019:", int(duplicates.sum()))
print("rows -> isic2018:", len(isic2018), "| isic2019:", len(isic2019))


## 2.4 Preprocessing and label encoding

Required: preprocessing, augmentation and label encoding. Produces: fixed class order, uint8 cache, `tf.data` pipeline, one-hot labels.

In [ ]:
# TODO(2.4): define CLASS_NAMES in fixed order (MEL, NV, BCC, AKIEC, BKL, DF, VASC) and the label maps.
# TODO(2.4): resize to the model input size and cache uint8 arrays on Drive to avoid re-decoding JPEGs.
# TODO(2.4): build tf.data datasets for train / validation / internal test with one-hot targets.

## 2.5–2.6 Augmentation

Required: show two random augmentations of one sample side by side; explain variance without label change. Produces: augmentation figure.

In [ ]:
# TODO(2.5): define the training-only augmentation pipeline as Keras layers.
# TODO(2.5): display the original image plus two independent random augmentations side by side.
# TODO(2.6): write how augmentation changes variance while labels stay unchanged.

## 2.7–2.9 Model, summary and feature maps

Required: build the model, print summary, extract feature maps, explain shapes and backpropagation. Produces: summary file, parameter counts, feature-map grid.

In [ ]:
# TODO(2.7): build ResNet-50 (ImageNet weights, include_top=False) + GAP + dropout + softmax head.
# TODO(2.7): print model.summary() into a text file and report total and trainable parameters.
# TODO(2.8): extract activations from one hidden layer for one test image and plot the channel grid.
# TODO(2.9): explain why spatial size shrinks and channel depth grows, and how backward updates weights.

## Part 1.1 support: gradient magnitude per layer, VGG vs ResNet

Required by 1.1: a graph of gradient magnitudes per layer for VGG vs ResNet. Produces: the Part 1 figure and its data table.

In [ ]:
# TODO(1.1): build a VGG and a ResNet with the same input size.
# TODO(1.1): run one forward and one backward pass, record the gradient norm of every layer.
# TODO(1.1): plot gradient magnitude against layer depth for both models and save the figure.

## 2.10–2.13 Training and learning curves

Required: adaptive optimizer, fixed epochs, loss curves with learning rate or gradient norm, overfitting analysis. Produces: training history and curves.

In [ ]:
# TODO(2.10): choose AdamW (adaptive optimizer) and a fixed number of epochs; state the learning-rate schedule.
# TODO(2.10): compute class weights from the training split only.
# TODO(2.11): log per-epoch training loss, validation loss, learning rate and gradient norm.
# TODO(2.12): plot train and validation loss with the learning rate on a second axis.
# TODO(2.13): mark where validation loss starts to rise and write the plateau and overfitting analysis.

## 2.14–2.17 Internal test evaluation

Required: test accuracy and loss, normalised confusion matrix, top-3 confident errors, error analysis. Produces: evaluation figures and tables.

In [ ]:
# TODO(2.14): evaluate the selected weights on the internal test split (accuracy and loss).
# TODO(2.15): plot the row-normalised 7x7 confusion matrix.
# TODO(2.16): show the three wrong predictions with the highest predicted-class probability.
# TODO(2.17): write the error analysis and check it refers to the current final image IDs.

## 2.18–2.20 External dataset, unseen image and calibration

Required: external evaluation, unseen-image probabilities, softmax discussion, calibration. Produces: external table, bar chart, reliability diagram and ECE.


In [ ]:
# TODO(2.18): evaluate on the ISIC 2019 external split (7-class mapping) and report the results table.
# TODO(2.19): pick one fixed unseen ISIC 2019 image, predict it and plot the 7-class bar chart.
# TODO(2.20): explain logits -> softmax -> probabilities, then plot the reliability diagram and report ECE.


## Artifacts, reproducibility and submission packaging

Required: one self-contained notebook and a verified results archive. Produces: weights, metrics JSON, all figures, results ZIP.

In [ ]:
# TODO: save selected weights, metrics JSON and every report figure with fixed file names.
# TODO: store seeds, versions and the split manifest inside the results archive.
# TODO: rerun this notebook top-to-bottom in a fresh Colab runtime before packaging.
# TODO: export the results ZIP and verify its contents.